## Vehicle Detection and Counting Using OpenCV

This project explores computer vision techniques for analysing traffic footage and detecting vehicles using Python and OpenCV. It demonstrates video frame processing, image analysis, and visualisation techniques for identifying vehicles in a traffic scene.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import math

## Loading and Displaying the First Frame

Before performing vehicle detection, the traffic video is loaded and an initial frame is displayed for visual inspection.

This step helps verify that the video has been loaded successfully and provides an overview of the traffic scene before further image processing and vehicle detection are performed.

In [ ]:
cap = cv2.VideoCapture("Traffic_Laramie_1.mp4")

ret, frame = cap.read()
print("Frame from original video :")

plt.figure(figsize = (10,6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

cap.release()

## Region of Interest (ROI) Selection

A Region of Interest (ROI) is defined to focus vehicle detection on the relevant road area while excluding unnecessary portions of the video frame.

The ROI boundaries are calculated relative to the frame dimensions, allowing a specific section of the traffic scene to be extracted for further processing.

The selected region is then displayed to visually verify that the appropriate road area has been captured.

In [ ]:
#Setting ROI 
h, w, _ = frame.shape

shift = int(w * 0.10) 
y1 = int(h * 0.35)
y2 = int(h * 0.95)
x1 = int(w * 0.05) + shift
x2 = int(w * 0.95) + shift
x1 = max(0, x1)
x2 = min(w, x2)

roi = frame[y1:y2, x1:x2]

plt.figure(figsize=(10,6))
plt.imshow(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
plt.axis("off")
print("ROI preview : ")
plt.show()

## Vehicle Detection Using Background Subtraction

This section implements a vehicle detection pipeline using OpenCV to identify moving objects within the selected Region of Interest (ROI).

# Detection Pipeline

Each video frame undergoes the following processing steps:

ROI Extraction: Extracts the selected road area to focus detection on relevant regions.

Gaussian Blur: Applies a Gaussian filter to reduce image noise and smooth the frame.

Background Subtraction: Uses OpenCV's MOG2 background subtraction algorithm to distinguish moving foreground objects from the background.

Morphological Processing: Applies morphological operations to reduce unwanted noise and improve the foreground mask.

Contour Filtering: Identifies candidate moving objects and filters them based on size and shape constraints.

This pipeline is applied to traffic footage to detect moving vehicles while reducing interference from background elements and smaller objects.

### Vehicle Detection — Traffic Video 1

The detection pipeline is applied to the first traffic video to identify moving vehicles within the selected road region.

In [ ]:
#video 1
cap = cv2.VideoCapture("Traffic_Laramie_1.mp4")

bg_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500, varThreshold=50, detectShadows=True
)

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

MIN_CONTOUR_AREA = 800
MIN_W, MIN_H = 20, 20
ASPECT_MIN, ASPECT_MAX = 0.3, 4.0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    roi = frame[y1:y2, x1:x2]
    roi_blur = cv2.GaussianBlur(roi, (5, 5), 0)

    fg_mask = bg_subtractor.apply(roi_blur)

    _, fg_mask = cv2.threshold(fg_mask, 200, 255, cv2.THRESH_BINARY)

    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel, iterations=1)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    roi_boxed = roi.copy()

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < MIN_CONTOUR_AREA:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        if w < MIN_W or h < MIN_H:
            continue

        aspect = w / float(h)
        if aspect < ASPECT_MIN or aspect > ASPECT_MAX:
            continue

        cv2.rectangle(roi_boxed, (x, y), (x+w, y+h), (0, 255, 0), 2)

    cv2.imshow("Foreground Mask (clean)", fg_mask)
    cv2.imshow("Cars detected (ROI)", roi_boxed)

    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


### Vehicle Detection — Traffic Video 2

The same detection pipeline is applied to a second traffic video to demonstrate vehicle detection on a different input sequence.

In [ ]:
#video 2
cap = cv2.VideoCapture("Traffic_Laramie_2.mp4")
bg_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500, varThreshold=50, detectShadows=True
)

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

MIN_CONTOUR_AREA = 800
MIN_W, MIN_H = 20, 20
ASPECT_MIN, ASPECT_MAX = 0.3, 4.0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    roi = frame[y1:y2, x1:x2]
    roi_blur = cv2.GaussianBlur(roi, (5, 5), 0)

    fg_mask = bg_subtractor.apply(roi_blur)

    _, fg_mask = cv2.threshold(fg_mask, 200, 255, cv2.THRESH_BINARY)

    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel, iterations=1)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    roi_boxed = roi.copy()

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < MIN_CONTOUR_AREA:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        if w < MIN_W or h < MIN_H:
            continue

        aspect = w / float(h)
        if aspect < ASPECT_MIN or aspect > ASPECT_MAX:
            continue

        cv2.rectangle(roi_boxed, (x, y), (x+w, y+h), (0, 255, 0), 2)

    cv2.imshow("Foreground Mask (clean)", fg_mask)
    cv2.imshow("Cars detected (ROI)", roi_boxed)

    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


## Results and Limitations

The implementation uses background subtraction and contour filtering to identify moving objects within a selected traffic region. Detected objects are highlighted using green bounding boxes, while the foreground mask provides a visual representation of moving regions.

The detection pipeline is applied to two traffic video inputs using the same processing techniques.

### Limitations

- Detection accuracy may be affected by changing lighting conditions, shadows, and background movement.
- Vehicles that overlap or travel close together may be detected as a single object.
- Contour size and aspect ratio thresholds may require adjustment for different camera angles and video resolutions.
- The current implementation focuses on vehicle detection and visualisation rather than persistent vehicle tracking or counting.

### Input Requirements

The notebook expects two traffic video files, `Traffic_Laramie_1.mp4` and `Traffic_Laramie_2.mp4`, in its working directory. These files are not included in the repository.

To reproduce the demonstration, provide compatible traffic videos and update the file paths if necessary.